###  Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import torch
from copy import deepcopy
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

from FusionTrainer import FusionTrainer
from TopoFeatExtractor import TopoFeatExtractor   
from LatentGateModel import LatentGateModel
from MeSHDataLoaders import load_ddi_data, create_folds
from TopoAblation import TopoAblation

In [2]:
# ⚙️ 2. Paths & Experiment Settings
graph_path = "networks/simon_network.graphml"
pos_path = "datasets/DDI_positive_pairs.csv"
neg_path = "datasets/DDI_negative_pairs.csv"
emb_dir = "datasets/"
extended_drug_categories_path = "datasets/extended_drug_info.csv"
protein_feat_path = "datasets/drug_GO_neighborhood_features.csv"

In [3]:
# Experiment hyperparameters
n_splits   = 5
batch_size = 128
epochs     = 50
lr         = 1e-3
weight_decay = 1e-5
seed       = 42
patience   = 5
num_experiments = 1
use_betweenness = False # Whether to use betweenness centrality as a topo feature --> It takes too long :(

### Load MeSH Data

In [4]:
# Load MeSH-based embeddings + DDI pairs for 'mid_level'
X, y, pairs = load_ddi_data(
    level="mid_level",
    pos_path=pos_path,
    neg_path=neg_path,
    emb_dir=emb_dir,
    seed=seed,
)
folds = create_folds(X, y, n_splits=n_splits, seed=seed)


[mid_level] Raw coverage:
   Positives: 453,524
   Negatives: 5,391
[mid_level] After embedding filtering:
   Total pairs: 458,915
   Positives: 453,524
   Negatives: 5,391
   ⚖️  Balancing AFTER filtering: 5391 pos & 5391 neg
[mid_level] FINAL dataset balanced: 10,782 pairs
   → 5391 pos, 5391 neg


In [5]:
# Load per-drug embeddings (for fusion model only) ---
emb_df = pd.read_csv(f"{emb_dir}/MeSH_mid_level_tfidf_svd128.csv", index_col=0)

In [6]:
# Load drug category mapping (low, mid, deep)
drug_to_cat = pd.read_csv(extended_drug_categories_path).set_index("drugbank_id")["knowledge_category"].to_dict()

### Get Trainer

In [7]:
# Load your trainer
trainer = FusionTrainer(
    fusion_model_wo_go=None,
    fusion_model_go=None,
    baseline_cls=None,
    topo_extractor=None,
    topo_extractor_with_go=None,
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=128,
    epochs=20,
    lr=1e-3,
    weight_decay=1e-5,
    patience=5,
    out_dir="results"
)

trainer.drug_to_cat = drug_to_cat # BAD DESIGN!!

FusionTrainer initialized.
   → Using GO extractor: False


### Load topo extractor

In [8]:
topo_extractor = TopoFeatExtractor(graph_path, 
                                    use_betweenness=use_betweenness,
                                    use_protein_neighborhood=False,
                                    protein_feat_path=None,
                                    use_weighted = False)

   Graph loaded: 17976 nodes, 268307 edges
   Found 3263 drug nodes.



### Topo-Only Ablation

In [9]:
ablation = TopoAblation(
    trainer = trainer,
    topo_extractor = topo_extractor,
)

In [10]:
ablation.run_topo_only_ablation(
    X=X,
    y=y,
    pairs=pairs,
    folds=folds,
    emb_df=emb_df,           
    drug_to_cat=drug_to_cat, 
    num_experiments=1
)


=== TOPO-ONLY ABLATION EXPERIMENT ===

=== Topo-Only Experiment 1/1 ===

--- Fold 1/5 ---
Removing 1088 eval DDI edges for leakage safety.
Computing topological features for 1667 drugs...
→ Processed 500/1667 drugs (13.6s elapsed)
→ Processed 1000/1667 drugs (30.3s elapsed)
→ Processed 1500/1667 drugs (45.8s elapsed)
Finished 1667 drugs in 50.6s total.
Computed features for 1667 drugs.

[Mode = full_topo]

[Mode = wo_degree]

[Mode = wo_cluster]

[Mode = wo_centrality]

--- Fold 2/5 ---
Removing 1102 eval DDI edges for leakage safety.
Computing topological features for 1667 drugs...
→ Processed 500/1667 drugs (14.8s elapsed)
→ Processed 1000/1667 drugs (33.7s elapsed)
→ Processed 1500/1667 drugs (51.3s elapsed)
Finished 1667 drugs in 56.5s total.
Computed features for 1667 drugs.

[Mode = full_topo]

[Mode = wo_degree]

[Mode = wo_cluster]

[Mode = wo_centrality]

--- Fold 3/5 ---
Removing 1059 eval DDI edges for leakage safety.
Computing topological features for 1667 drugs...
→ Proce

### Plot per-category results

In [1]:
import pickle

with open("results/ablation/topo_only_results.pkl", "rb") as f:
    topo_results = pickle.load(f)

print("Loaded topo-only results.")

Loaded topo-only results.


In [14]:
def get_topo_modes(topo_results):
    exp0 = list(topo_results.keys())[0]
    fold0 = list(topo_results[exp0].keys())[0]
    modes = list(topo_results[exp0][fold0].keys())
    print("Detected topo-only modes:", modes)
    return modes

modes = get_topo_modes(topo_results)

Detected topo-only modes: ['full_topo', 'wo_degree', 'wo_cluster', 'wo_centrality']


In [15]:
import os
import numpy as np
import matplotlib.pyplot as plt

def plot_topo_only_per_category(topo_results, save_dir="results/ablation/topo_only_plots"):
    os.makedirs(save_dir, exist_ok=True)

    # Detect modes and categories
    exp0 = list(topo_results.keys())[0]
    fold0 = list(topo_results[exp0].keys())[0]

    modes = list(topo_results[exp0][fold0].keys())
    categories = list(topo_results[exp0][fold0][modes[0]].keys())

    metrics = ["auc", "acc", "f1", "precision", "recall"]

    for cat in categories:

        fig, axs = plt.subplots(1, 5, figsize=(22, 4))
        fig.suptitle(f"Topo-Only Ablation — Category: {cat}", fontsize=16)

        for j, metric in enumerate(metrics):
            means, stds = [], []

            for mode in modes:
                vals = []
                for exp in topo_results:
                    for fold in topo_results[exp]:
                        vals.append(topo_results[exp][fold][mode][cat][metric])
                vals = np.array(vals)
                means.append(vals.mean())
                stds.append(vals.std())

            axs[j].bar(modes, means, yerr=stds, capsize=4)
            axs[j].set_title(metric)
            axs[j].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        fig.savefig(os.path.join(save_dir, f"topo_only_{cat}.png"), dpi=150)
        plt.close(fig)

    print(f"Saved per-category topo-only plots in: {save_dir}")

In [16]:
plot_topo_only_per_category(topo_results)

Saved per-category topo-only plots in: results/ablation/topo_only_plots


### Plot aggregated summary

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt

def plot_topo_only_summary_per_metric(
    topo_results,
    save_dir="results/ablation/topo_only_summary_plots"
):
    """
    Saves *separate* barplots for each metric:
    auc, acc, f1, precision, recall.
    """

    metrics = ["auc", "acc", "f1", "precision", "recall"]

    # detect ablation modes dynamically
    exp0 = list(topo_results.keys())[0]
    fold0 = list(topo_results[exp0].keys())[0]
    modes = list(topo_results[exp0][fold0].keys())

    os.makedirs(save_dir, exist_ok=True)

    for metric in metrics:
        means, stds = [], []

        for mode in modes:
            vals = []
            for exp in topo_results:
                for fold in topo_results[exp]:
                    for cat in topo_results[exp][fold][mode]:
                        vals.append(
                            topo_results[exp][fold][mode][cat][metric]
                        )
            vals = np.array(vals)
            means.append(vals.mean())
            stds.append(vals.std())

        # ---- Make a separate plot for this metric ----
        plt.figure(figsize=(7,4))
        plt.bar(modes, means, yerr=stds, capsize=4)
        plt.title(f"Topo-Only Ablation — {metric.upper()}")
        plt.ylabel(metric.upper())
        plt.xticks(rotation=45)
        plt.tight_layout()

        out_path = os.path.join(save_dir, f"summary_{metric}.png")
        plt.savefig(out_path, dpi=150)
        plt.close()

        print(f"Saved: {out_path}")

In [4]:
plot_topo_only_summary_per_metric(topo_results)

Saved: results/ablation/topo_only_summary_plots/summary_auc.png
Saved: results/ablation/topo_only_summary_plots/summary_acc.png
Saved: results/ablation/topo_only_summary_plots/summary_f1.png
Saved: results/ablation/topo_only_summary_plots/summary_precision.png
Saved: results/ablation/topo_only_summary_plots/summary_recall.png
